In [ ]:
import pandas as pd 
from plotnine import *
import numpy as np
from vpop_calibration import *

%load_ext autoreload
%autoreload 2

In [ ]:
df = pd.read_csv("Mavoglurant_Benchmark_Dataset.csv")


obs_df = (
    df.loc[df["EVID"] == 0]
      .rename(
          columns={
              "ID": "id",
              "TIME": "time",
              "DV": "value",
              "DOSE": "dose"

          }
      )
      [["id", "time", "value", "dose", "WT"]]
      .astype({"value": "float"})
)
obs_df["time"] = obs_df["time"].apply(lambda t: t * 60 * 60)
obs_df["value"]= obs_df["value"].apply(lambda t: np.log(t))
obs_df["output_name"] = "logC15"
obs_df["protocol_arm"] = "identity"
display(df.head())
display(obs_df.head())

## Reference model

In [ ]:
model = SimworkModelBinding(
    path_to_model="cm.json",
    path_to_solving_options="sv.json",
    inputs=["KbBR", "CLint","KbMU","KbAD","KbBO","KbRB","dose","WT"],
    outputs=["logC15"],
)
print(model.inputs)

struct_model = StructuralSimwork(model=model)

## Generate training data

In [ ]:

protocol_design = pd.DataFrame({
    "protocol_arm": ["identity"]
})

struct_model = StructuralSimwork(
    model=model,
    protocol_design=protocol_design,
)

param_ranges = {
    "dose": {"low": 20, 
            "high": 50, 
            "log": False,
        },
    "WT": {"low": 40, 
           "high": 120,
           "log": False,
    },
    "CLint": {
        "low": 6.89 - 2 * 0.288,
        "high": 6.89 + 2 * 0.288,
        "log": True,
    },
    "KbBR": {
        "low": 1.49 - 2 * 1.72,
        "high": 1.49 + 2 * 1.72,
        "log": True,
    },
    "KbMU": {
        "low": 1.43 - 2 * 1.07,
        "high": 1.43 + 2 * 1.07,
        "log": True,
    },
    "KbBO": {
        "low": 0.65 - 2 * 1.86,
        "high": 0.65 + 2 * 1.86,
        "log": True,
    },
    "KbAD": {
        "low": 2 - 2 * 7.33,
        "high": 2 + 2 * 7.33,
        "log": True,
    },
    "KbRB": {
        "low": 1.87 - 2 * 0.386,
        "high": 1.87 + 2 * 0.386,
        "log": True,
    },
}

In [ ]:
time_span = (0, 48*60*60)
nb_steps = 100
time_steps = np.linspace(*time_span, nb_steps).tolist()
dataset = generate_training_data(
    struct_model=struct_model,
    ranges=param_ranges,
    log_nb_ind=5,
    time=time_steps,
)

## Define and train GP model

In [ ]:
learned_ode_params = list(param_ranges.keys())
descriptors = learned_ode_params + ["time"]
print(descriptors)

# initiate our GP class
myGP = GP(
    dataset,
    descriptors,
    var_strat="IMV",  # either IMV (Independent Multitask Variational) or LMCV (Linear Model of Coregionalization Variational)
    kernel="RBF",  # Either RBF or SMK
    nb_inducing_points=100,
    mll="ELBO",  # default, otherwise PLL
    nb_training_iter=1000,
    training_proportion=0.7,
    learning_rate=0.1,
    lr_decay=0.99,
    jitter=1e-6,
    log_inputs=learned_ode_params,
    log_outputs=[],
    plot_frame_rate=50,
)

In [ ]:
myGP.train()

In [ ]:
myGP.eval_perf()
myGP.plot_obs_vs_predicted(data_set="training")

## Optimize the GP surrogate using SAEM


In [ ]:
structural_gp = StructuralGp(myGP)
# NLME model parameters
prior = {
    "pdu": {
        "CLint": {"prior": np.exp(7.6), "prior_omega": 0.5},
        "KbBR": {"prior": np.exp(0.5),"prior_omega": 0.25},
        "KbMU": {"prior": np.exp(0.3),"prior_omega": 0.25}, 
        "KbBO": {"prior": np.exp(0.03),"prior_omega": 0.25},
        "KbAD":  {"prior": np.exp(2), "prior_omega": 0.25},
        "KbRB": {"prior": np.exp(0.3), "prior_omega": 0.25},
    },
    "pdk": {
        "WT",
        "dose",
    },
    "error_model": {
        "logC15": {"error_type": "additive", "sigma": 0.5},
    },
}
config = Config(
    saem=SaemConfigDict(
        nb_phase1_iterations=300,
        nb_phase2_iterations=200,
        plot_frames=5,
        optim_max_fun=2,
        verbose=False,
        mcmc_first_burn_in=0,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)

nlme_surrogate = NlmeModel(df=obs_df, prior_params=prior, structural_model=structural_gp, config=config)
nlme_surrogate.optimizer.run()   

In [ ]:
nlme_surrogate.diagnostics.compute_ebe(max_iter=10)

In [ ]:
nlme_surrogate.plot.map_estimates()
nlme_surrogate.plot.map_estimates_gof()
    